In [3]:
##### Converts raw raster on night lights into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # % of area with night lights

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats

In [4]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# night lights raster
lights = f"{cd}/Data/Raw/Predictors/Night_lights_Li/Harmonized_DN_NTL_2020_simVIIRS.tif"

# save paths
pixels_binary = f"{cd}/Data/Raw/Predictors/Night_lights_Li/nightlights_2020_binary.tif"
pixels = f"{cd}/Data/Clean/Predictors/Rasters/share_with_nightlights.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/share_with_nightlights.csv"

In [5]:
with rasterio.open(lights) as src:
    lights_raw = src.read(1).astype(np.float32)
    lights_meta = src.meta.copy()
    lights_nodata = src.nodata

# Mask nodata
lights_raw[lights_raw == lights_nodata] = np.nan

# Binary: 1 = nightlights > 0, 0 = nightlights == 0, nan = nodata
lights_binary = np.where(
    ~np.isnan(lights_raw) & (lights_raw > 0), 1,
    np.where(~np.isnan(lights_raw) & (lights_raw == 0), 0, np.nan)
)

# Save
out_meta = lights_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = lights_binary.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_binary, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

Pixels with electricity:    50,113,243
Pixels without electricity: 675,706,758
% with access: 6.9%


In [6]:
#### Run resampling for pixel scale 

### PREP 

# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()

# set details for rasters
rasters_to_resample = {
    "lights": (pixels_binary, pixels)
}

### RESAMPLE
for name, (src_path, out_path) in rasters_to_resample.items():
    with rasterio.open(src_path) as src:
        src_nodata = src.nodata

        dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.average,
            src_nodata=src_nodata,
            dst_nodata=np.nan,
        )

    out_meta = dst_meta.copy()
    out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

    out_arr = dst_array.copy()
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_arr, 1)

In [7]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
with rasterio.open(pixels_binary) as src:
    raster_crs = src.crs

gdf_proj = total_geo.to_crs(raster_crs)

# set final form 
result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE
for col_name, rpath in [("share", pixels)]:
    zonal = rasterstats.zonal_stats(
        gdf_proj,
        rpath,
        stats="mean",
        nodata=-9999
    )
    result[col_name] = [z["mean"] for z in zonal]

### SAVE
result.to_csv(vectors, index=False)